# CSCI 3151 â€” Foundations of Machine Learning

### M33 â€” Deep Networks & Gradient Flow (mini-lab)

In this short in-class lab, you'll poke at gradient flow in small multilayer perceptrons trained on a simple 2D classification problem. We'll compare shallow vs deep networks and different activations by looking directly at the norms of the gradients layer by layer after a backward pass.

We'll treat the data as coming from a noisy non-linear decision boundary (a â€œtwo moonsâ€ toy dataset). Our main question:

> How do depth, activation choice, and initialization scaling affect how much gradient signal reaches the early layers?

If gradient flow is unhealthy, deep networks can enter a **silent failure** mode where only the last few layers learn anything, even though the loss is still changing.

**This mini-lab reinforcesâ€¦**

- the computational picture of **gradient flow as a product of local derivatives** from Modules M31 & M33,
- practical intuition for **vanishing and exploding gradients** in deep MLPs,
- reading and interpreting **per-layer gradient-norm plots** as a debugging tool,
- using **PyTorch autograd** to inspect and reason about training dynamics.


## Assumptions & scope

- You are comfortable with basic Python, NumPy, and plotting with Matplotlib.
- You have already seen feedforward neural networks, backpropagation, and PyTorch's autograd interface (`loss.backward()`, `.grad`, `optimizer.step()`) in earlier modules.
- We work with a small synthetic 2D dataset and small fully-connected networks, ignoring issues like regularization and train/validation splitting so we can focus on gradient flow.
- We will not derive singular values or Jacobians in full detail here; instead we connect the *shape* of gradient-norm curves to the theory you saw in the video.

Feel free to experiment beyond the guided cells if you have extra time.


In [ ]:
# Setup: imports and global settings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import make_moons

# Make plots a bit nicer
plt.style.use("default")

# Reproducibility
SEED = 3151
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Create a simple non-linear 2D dataset

X_np, y_np = make_moons(n_samples=1024, noise=0.2, random_state=SEED)

# Convert to PyTorch tensors
X = torch.from_numpy(X_np).float().to(device)
y = torch.from_numpy(y_np).long().to(device)

# We'll use a fixed mini-batch for our gradient experiments
batch_X = X[:256]
batch_y = y[:256]

print(f"Full dataset shape: X={X.shape}, y={y.shape}")
print(f"Mini-batch shape: batch_X={batch_X.shape}, batch_y={batch_y.shape}")

# Quick visualization (on CPU)
plt.figure(figsize=(4, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, s=8, alpha=0.7)
plt.title("Two-moons synthetic dataset")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()


In [ ]:
# Define a simple fully-connected network and gradient utilities

class SimpleMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, depth=3, activation="tanh"):
        super().__init__()
        layers = []
        dims = [in_dim] + [hidden_dim] * (depth - 1) + [out_dim]
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
        self.layers = nn.ModuleList(layers)

        if activation == "tanh":
            self.activation = torch.tanh
        elif activation.lower() == "relu":
            self.activation = F.relu
        else:
            raise ValueError(f"Unsupported activation: {activation}")

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            # Apply nonlinearity after all but the last layer
            if i < len(self.layers) - 1:
                x = self.activation(x)
        return x

loss_fn = nn.CrossEntropyLoss()


def one_backward_pass(depth, activation="tanh", hidden_dim=64, lr=0.1):
    """
    Build a fresh network, run a single forward+backward step on batch_X / batch_y,
    and return the loss and a list of gradient norms (one per layer weight matrix).
    """
    # Reset seed so networks with the same hyperparameters are comparable
    torch.manual_seed(SEED)

    model = SimpleMLP(in_dim=2,
                      hidden_dim=hidden_dim,
                      out_dim=2,
                      depth=depth,
                      activation=activation).to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    optimizer.zero_grad()
    logits = model(batch_X)
    loss = loss_fn(logits, batch_y)
    loss.backward()

    grad_norms = []
    for layer in model.layers:
        if layer.weight.grad is None:
            grad_norms.append(0.0)
        else:
            grad_norms.append(layer.weight.grad.norm().item())

    # We don't actually need to take an optimizer step for these diagnostics,
    # but doing so would be fine.

    return loss.item(), grad_norms


## Core Task 1 â€“ Gradient norms vs depth (tanh networks)

In the video for M33 you saw that gradients at early layers are obtained by multiplying together many local Jacobians. Even if each layer is only slightly contractive, the product can shrink rapidly with depth, leading to **vanishing gradients**.

In this task we'll hold everything fixed except the depth of the network and inspect how the Frobenius norms \(\lVert \partial L / \partial W_\ell \rVert_F\) change across layers.

**Your goals:**

- Run the experiment below for several depths (e.g., 3, 10, 30) using `tanh` activations.
- Look for systematic patterns in the gradient norms as you move from the output layer back toward the input.
- Note at what depth you start to see clear signs of vanishing gradients.

After running the code cell, jot down a few observations in the markdown cell that follows.


In [ ]:
# Experiment: gradient norms across depth for tanh networks
#
# TODO 1:
#   Change `depths` to include at least one *shallower* and one *deeper* network.
#   Run the cell again and see how the gradient norms behave.
#
# TODO 2:
#   Read through the loop below and make sure you understand how `one_backward_pass`
#   is being used. As a selfâ€‘check, try rewriting the body of the loop yourself
#   (without looking) in your own copy of the notebook.

depths = [3, 10, 30]  # TODO: edit this list

hidden_dim = 64

grad_profiles = {}
losses = {}

for d in depths:
    # TODO (optional but recommended): reâ€‘implement this line from memory
    loss, norms = one_backward_pass(depth=d, activation="tanh", hidden_dim=hidden_dim)

    grad_profiles[d] = norms
    losses[d] = loss

    print(f"Depth {d:2d} â€” loss = {loss:.3f}, gradient norms (per layer):")
    print(norms)
    print()

# Plot gradient norms for each depth on a log scale
plt.figure(figsize=(7, 4))
for d, norms in grad_profiles.items():
    layers = np.arange(1, len(norms) + 1)
    plt.plot(layers, norms, marker="o", label=f"depth={d}")

plt.yscale("log")
plt.xlabel("Layer index (1 = closest to input)")
plt.ylabel("||âˆ‚L/âˆ‚W_l|| (Frobenius norm, log scale)")
plt.title("Gradient norms across depth for tanh networks")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.tight_layout()
plt.show()

# Optional: tabulate the norms for easier comparison
max_len = max(len(v) for v in grad_profiles.values())
data = {}
for d, norms in grad_profiles.items():
    arr = np.full(max_len, np.nan)
    arr[: len(norms)] = norms
    data[f"depth={d}"] = arr

df_gradients = pd.DataFrame(data, index=[f"Layer {i}" for i in range(1, max_len + 1)])
df_gradients


ðŸ“ Core Task 1 â€“ Your observations
    
Use this space to summarize what you see in the plots/table above. For example:
    
- How do the gradient norms near the input layer compare to those near the output layer?
- How does increasing depth affect the shape of the gradient-norm curve?
- At what depth (if any) do you feel gradient flow is becoming â€œunhealthyâ€? Why?
    
(Bullet points are totally fine.)


## Core Task 2 â€“ Activation choice: tanh vs ReLU

Activation functions change the typical size of derivatives at each layer. Roughly:

- `tanh` squashes activations into \((-1, 1)\), and its derivative is often **smaller than 1** in magnitude.
- `ReLU` has derivative either 0 or 1, so when it's active it does **not shrink** the gradient.

Here we'll fix the depth and compare gradient norms for `tanh` and `ReLU` networks with the same architecture and learning rate.

**Your goals:**

- Run the cell below for a reasonably deep network (e.g., depth 30).
- Compare where the tanh vs ReLU curves sit and how fast they change as you move toward the input layer.
- Relate what you see to the â€œproducts of Jacobiansâ€ story from the lecture.


In [ ]:
# Experiment: tanh vs ReLU for a fixed depth
#
# TODO 1:
#   Change `depth` and `hidden_dim` and see how the comparison changes.
# TODO 2:
#   Modify the plotting code so that it uses different markers/colours or
#   plots the *log10* of the norms instead of using `plt.yscale("log")`.

depth = 30      # TODO: try values like 3, 10, 50
hidden_dim = 64 # TODO: try making this wider or narrower

# Run a single backward pass for each activation
loss_tanh, norms_tanh = one_backward_pass(depth=depth, activation="tanh", hidden_dim=hidden_dim)
loss_relu, norms_relu = one_backward_pass(depth=depth, activation="relu", hidden_dim=hidden_dim)

print(f"tanh: depth={depth}, loss={loss_tanh:.3f}")
print(f"ReLU: depth={depth}, loss={loss_relu:.3f}")

layers = np.arange(1, depth + 1)

plt.figure(figsize=(7, 4))
plt.plot(layers, norms_tanh, marker="o", label="tanh")
plt.plot(layers, norms_relu, marker="s", label="ReLU")

plt.yscale("log")
plt.xlabel("Layer index (1 = closest to input)")
plt.ylabel("||âˆ‚L/âˆ‚W_l|| (Frobenius norm, log scale)")
plt.title(f"Gradient norms for depth={depth}: tanh vs ReLU")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.tight_layout()
plt.show()


ðŸ“ Core Task 2 â€“ Your observations

A few prompts:

- Which activation seems to preserve gradient signal better to early layers in this setup?
- Does one activation show signs of vanishing or exploding gradients more clearly?
- How does this match your expectations from the theory? Any surprises?


## Stretch Task â€“ Play with initialization scale

Another lever that affects gradient flow is the **scale of the initial weights**. If all weights are tiny, each local Jacobian tends to shrink vectors; if all weights are huge, Jacobians can blow up norms.

Below we add a helper that rescales all linear weights by a constant factor *after* PyTorch's default initialization. You can then compare gradient norms for different scales.

**Ideas to try (if you have time):**

- Fix a depth (e.g., 20 or 30) and activation (e.g., ReLU), and compare scales like `0.1`, `1.0`, and `5.0`.
- See whether you can deliberately create an exploding-gradient situation.
- Try a â€œgoodâ€ scale where gradients are reasonably balanced across layers.


In [ ]:
# Helper: rescale weights and measure gradient norms

def one_backward_with_rescale(depth, activation="relu", hidden_dim=64, lr=0.1, scale=1.0):
    torch.manual_seed(SEED)
    model = SimpleMLP(in_dim=2,
                      hidden_dim=hidden_dim,
                      out_dim=2,
                      depth=depth,
                      activation=activation).to(device)

    # Rescale all linear weights by a constant factor (in-place, without tracking gradients)
    if scale != 1.0:
        with torch.no_grad():
            for layer in model.layers:
                layer.weight.mul_(scale)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    optimizer.zero_grad()
    logits = model(batch_X)
    loss = loss_fn(logits, batch_y)
    loss.backward()

    grad_norms = [layer.weight.grad.norm().item() for layer in model.layers]
    return loss.item(), grad_norms


# Example experiment: play with scales for a fixed depth & activation
depth = 20
hidden_dim = 64
activation = "relu"
scales = [0.1, 1.0, 5.0]

grad_profiles_scaled = {}

for s in scales:
    loss, norms = one_backward_with_rescale(depth=depth,
                                            activation=activation,
                                            hidden_dim=hidden_dim,
                                            scale=s)
    grad_profiles_scaled[s] = norms
    print(f"Scale {s:>4} â€” loss = {loss:.3f}")

layers = np.arange(1, depth + 1)

plt.figure(figsize=(7, 4))
for s, norms in grad_profiles_scaled.items():
    plt.plot(layers, norms, marker="o", label=f"scale={s}")

plt.yscale("log")
plt.xlabel("Layer index (1 = closest to input)")
plt.ylabel("||âˆ‚L/âˆ‚W_l|| (Frobenius norm, log scale)")
plt.title(f"Gradient norms for different init scales (depth={depth}, activation={activation})")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.tight_layout()
plt.show()


## Wrap-up â€“ So what did we learn?

Take a minute to connect these experiments back to the big-picture story of gradient flow in deep networks.

- When did gradients look **healthy** across layers? What combination of depth, activation, and initialization created that behaviour?
- When did you see clear signs of **vanishing** or **exploding** gradients?
- How might tools like these gradient-norm plots help you debug a misbehaving deep network in the wild?

If you finish early, try designing your own experiment (e.g., change the hidden width, learning rate, or dataset) and see how the gradient profiles change.
